In [7]:
from IPython.display import Markdown, display
import requests
import pandas as pd
import plotly.express as px
import time

from plotly.subplots import make_subplots
import plotly.graph_objects as go
sparql_endpoint = "https://artresearch.net/sparql"
headers = {"Accept": "application/sparql-results+json", "Cache-Control": "no-cache", "Pragma": "no-cache"}

institutions = ["frick", "hertziana", "khi", "marburg", "pmc", "rkd", "warburg", "zeri"]

fig3 = make_subplots(rows=4, cols=2, subplot_titles=[i.upper() for i in institutions])

for idx, inst in enumerate(institutions):
    query = f"""
    PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
    PREFIX pharos-meta: <https://artresearch.net/resource/pharos/vocab/meta/>

    SELECT ?artist (SAMPLE(?name) AS ?label) (COUNT(DISTINCT ?work) AS ?count)
    WHERE {{
      ?work crm:P108i_was_produced_by ?production ;
            crm:P70i_is_documented_in <https://artresearch.net/resource/e31/{inst}> .
      ?production crm:P14_carried_out_by ?artist .
      OPTIONAL {{
        ?artist crm:P1_is_identified_by ?app .
        ?app crm:P2_has_type/crm:P127_has_broader_term* pharos-meta:preferred_name ;
            crm:P190_has_symbolic_content ?name .
      }}
    }}
    GROUP BY ?artist
    ORDER BY DESC(?count)
    LIMIT 10
    """
    r = requests.get(sparql_endpoint, params={"query": query, "_": int(time.time())}, headers=headers)
    if r.status_code != 200:
        print(f"{inst} failed")
        continue

    bindings = r.json()['results']['bindings']
    if not bindings:
        print(f"{inst}: no results")
        continue

    df = pd.DataFrame([{
        "artist": b.get("label", b["artist"])["value"] if "label" in b else b["artist"]["value"].split("/")[-1],
        "count": int(b["count"]["value"])
    } for b in bindings]).sort_values("count")

    row, col = idx // 2 + 1, idx % 2 + 1
    fig3.add_trace(go.Bar(x=df["count"], y=df["artist"], orientation="h", name=inst), row=row, col=col)

fig3.update_layout(height=1200, title_text="Top 10 Artists per Institution", showlegend=False)
fig3.show()